In [1]:
import importlib
import rasterio
import Utils 
import numpy as np
import Constants
import ConstantObjects
importlib.reload(Utils)
importlib.reload(Constants)
importlib.reload(ConstantObjects)
import inspect
import os

from rasterio.mask import mask


 




[Line 12] n_cols in ConstantObjects: 21
[Line 14] n_cols: 21, n_rows: 18
[Line 25] n_cols in ConstantObjects: 21
[Line 27] gdf_tree_circles.shape: (378, 1)
x0 =  -10783454.50783975
y0 =  2246774.099946275
draw_grid_box : dx =  5
draw_grid_box : dy =  4
x1,y1 etc
-10783504.354768038 2246750.855941879
-10783527.012462713 2246740.2904853355
-10783520.250570524 2246725.789560743
-10783497.59287585 2246736.3550172867
📍 Center of box: (19.777898, -96.869939)
x0 =  -10783454.50783975
y0 =  2246774.099946275
draw_grid_box : dx =  5
draw_grid_box : dy =  4
x1,y1 etc
-10783477.165534426 2246763.5344897313
-10783499.823229102 2246752.9690331877
-10783479.53755254 2246709.46625941
-10783456.879857862 2246720.0317159537
📍 Center of box: (19.777882, -96.869634)
Constants.n_cols =  21
Constants.n_rows =  18
x0 =  -10783454.50783975
y0 =  2246395.5503548854
draw_grid_box : dx =  5
draw_grid_box : dy =  4
x1,y1 etc
-10783454.50783975 2246395.5503548854
-10782194.50783975 2246395.5503548854
-10782194.50

In [2]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
rows = []  # collect per-date stats here
# Villanueva, VER
startDate = "2025-01-12"
endDate = "2025-08-16"
#bogus
startDate = "2021-10-01"
endDate = "2022-06-30"

dates = Utils.generate_date_range(startDate, endDate,  "%Y-%m-%d", 5)
#datesIncrementDaily = Utils.generate_date_range(startDate, endDate,  "%Y-%m-%d", 1)
print(f"[Line {inspect.currentframe().f_lineno}] ...   ")

precip_df = Utils.daily_precip_openmeteo(Constants.lat0, Constants.lon0, startDate, endDate)
print(f"[Line {inspect.currentframe().f_lineno}] ...   ")

precip_df.head()
print(f"[Line {inspect.currentframe().f_lineno}] ...   ")
                                 
#dates = Utils.generate_date_range("2025-06-16", "2025-06-17",  "%Y-%m-%d", 5)

# jan 12, no cloud directly over farm.
#Feb 11, lots of clouds over farm
ndmi_tile_tif=""
for date in dates:
    print(f"[Line {inspect.currentframe().f_lineno}] ... date = ",date)
    ndmi_tile_tif = Utils.make_path_name("s2_derived_tifs", Utils.latlon_to_s2_tile(Constants.lat0, Constants.lon0),date, "ndmi","tif")
    rgb_tile_tif = Utils.make_path_name("s2_derived_tifs", Utils.latlon_to_s2_tile(Constants.lat0, Constants.lon0),date, "rgb.uint8","tif")

    print(f"[Line {inspect.currentframe().f_lineno}] ... ndmi_tile_tif = ",ndmi_tile_tif)
    gdf_box = ConstantObjects.gdf_box_terreno_casa
    ndmi_terreno_casa_tif = Utils.make_path_name("s2_parcel_tifs", Utils.latlon_to_s2_tile(Constants.lat0, Constants.lon0),date, "ndmi-terreno-casa","tif")
    ndmi_plots_4_5_tif = Utils.make_path_name("s2_parcel_tifs", Utils.latlon_to_s2_tile(Constants.lat0, Constants.lon0),date, "ndmi-plots-4-5","tif")

    rgb_terreno_casa_tif = Utils.make_path_name("s2_parcel_tifs", Utils.latlon_to_s2_tile(Constants.lat0, Constants.lon0),date, "rgb-terreno-casa","tif")

    print(f"[Line {inspect.currentframe().f_lineno}] ... ndmi_terreno_casa_tif = ",ndmi_terreno_casa_tif)
    print(f"[Line {inspect.currentframe().f_lineno}] ... calling Utils.crop_ndmi_to_gdf(ndmi_tile_tif, ConstantObjects.gdf_box_terreno_casa, ndmi_terreno_casa_tif) ")
    stats_terreno_casa = Utils.crop_ndmi_to_gdf(ndmi_tile_tif, ConstantObjects.gdf_box_terreno_casa, ndmi_terreno_casa_tif) 
    print("Avg NDMI over AOI terreno casa:", stats_terreno_casa["mean"])
    stats_plots_4_5 = Utils.crop_ndmi_to_gdf(ndmi_tile_tif, ConstantObjects.gdf_box_plots_4_5, ndmi_plots_4_5_tif) 
    print("Avg NDMI over AOI plots 4,5:", stats_plots_4_5["mean"])

    print(f"[Line {inspect.currentframe().f_lineno}] ...  ")
    
    cloud_terreno_casa = Utils.estimate_cloud_fraction_rgb_over_gdf(rgb_tile_tif, ConstantObjects.gdf_box_terreno_casa,treat_zero_as_nodata=True)

    # Over your farm AOI:
    cloud_green = Utils.cloud_fraction_green_ratio_uint8(
        rgb_tile_tif,
        ConstantObjects.gdf_box_terreno_casa,  # or None for whole tile
        threshold=0.4,   # try 0.56–0.62 to tune
        v_min=0.08        # ignore very dark shadows
    )
    print(cloud_green)

    
    print(f"[Line {inspect.currentframe().f_lineno}] ... cloud_terreno_casa[cloud_frac]  ",cloud_terreno_casa["cloud_frac"])
    print(f"[Line {inspect.currentframe().f_lineno}] ... date  ",date)

    #cloud_plots_4_5 = Utils.estimate_cloud_fraction_rgb_over_gdf(rgb_tile_tif, ConstantObjects.gdf_box_plots_4_5,treat_zero_as_nodata=True)

    rows.append({
        "date": date,
        "ndmi_terreno_casa": stats_terreno_casa.get("mean", np.nan),
        "ndmi_plots_4_5":   stats_plots_4_5.get("mean",   np.nan),
        "cloud_frac_terreno_casa": cloud_terreno_casa.get("cloud_frac", np.nan),
        #"cloud_frac_terreno_casa": cloud_terreno_casa["cloud_frac"],
        #"cloud_frac_plots_4_5":    cloud_plots_4_5["cloud_frac"],
    })    
    Utils.show_ndmi_moist(ndmi_terreno_casa_tif, scale=12)
    #if os.path.exists(rgb_tile_tif):
    print(f"[Line {inspect.currentframe().f_lineno}] ... calling Utils.crop_ndmi_to_gdf(rgb_tile_tif, ConstantObjects.gdf_box_terreno_casa, rgb_terreno_casa_tif)  ")
    #####Utils.crop_ndmi_to_gdf(rgb_tile_tif, ConstantObjects.gdf_box_terreno_casa, rgb_terreno_casa_tif) 

    #Utils.crop_ndmi_to_gdf(rgb_tile_tif, ConstantObjects.gdf_box_terreno_casa, rgb_terreno_casa_tif)
    #if stats:
    #print("Avg NDMI over AOI:", stats["mean"])
    
    Utils.show_tif_fit(rgb_terreno_casa_tif, scale=12, cmap="gray")
    print(f"[Line {inspect.currentframe().f_lineno}] ... ")
    #else :
    #    print(f"[Line {inspect.currentframe().f_lineno}] The file rgb_tile_tif = ",rgb_tile_tif, " does not exist for date ",date,". Skipping this one.")





import matplotlib.pyplot as plt
import pandas as pd

# --- existing: rows -> df (sparse NDMI/Cloud dates)
df = pd.DataFrame(rows)
df["date"] = pd.to_datetime(df["date"]).dt.normalize()
df = df.sort_values("date")


    

[Line 13] ...   
[Line 1300] ...   
[Line 1305] ...   
[Line 1308] ... pd.to_datetime(d[time])   DatetimeIndex(['2021-10-01', '2021-10-02', '2021-10-03', '2021-10-04',
               '2021-10-05', '2021-10-06', '2021-10-07', '2021-10-08',
               '2021-10-09', '2021-10-10',
               ...
               '2022-06-21', '2022-06-22', '2022-06-23', '2022-06-24',
               '2022-06-25', '2022-06-26', '2022-06-27', '2022-06-28',
               '2022-06-29', '2022-06-30'],
              dtype='datetime64[ns]', length=273, freq=None)
[Line 1309] ... d[precipitation_sum]   [0.9, 0.8, 7.6, 3.6, 8.8, 2.2, 0.0, 0.9, 1.2, 0.0, 0.4, 0.1, 0.0, 0.0, 0.9, 1.8, 41.9, 7.0, 1.6, 3.5, 6.7, 6.4, 1.5, 1.6, 0.5, 0.0, 3.4, 11.6, 3.9, 0.2, 0.9, 1.7, 1.4, 3.3, 0.1, 9.4, 3.9, 8.5, 8.6, 12.9, 1.0, 0.2, 2.2, 3.3, 4.4, 0.0, 0.5, 0.0, 1.2, 49.4, 2.9, 2.0, 6.3, 23.0, 0.4, 0.4, 4.7, 1.9, 0.1, 3.1, 0.7, 0.0, 1.6, 0.3, 0.1, 0.0, 0.0, 0.2, 0.0, 0.0, 0.0, 0.0, 24.0, 0.1, 0.0, 0.0, 0.0, 0.0, 0.0, 5.8, 1.2, 2

AttributeError: module 'Utils' has no attribute 'estimate_cloud_fraction_rgb_over_gdf'

In [4]:


#Borana University, Ethiopia
#startDate = "2021-10-01"
#endDate = "2022-06-30"


import matplotlib.pyplot as plt
import pandas as pd

# --- existing: rows -> df (sparse NDMI/Cloud dates)
df = pd.DataFrame(rows)
df["date"] = pd.to_datetime(df["date"]).dt.normalize()
df = df.sort_values("date")


# --- existing: daily precip
precip_df = Utils.daily_precip_openmeteo(Constants.lat0, Constants.lon0, startDate, endDate)
p = precip_df.copy()
p["date"] = pd.to_datetime(p["date"]).dt.normalize()
p = p.sort_values("date")

# --- plot: NDMI (left), cloud (right), DAILY precip on a second right axis
fig, ax = plt.subplots(figsize=(10, 4))

# NDMI lines (sparse dates)
ax.plot(df["date"], df["ndmi_terreno_casa"], marker="o", label="NDMI terreno")
ax.plot(df["date"], df["ndmi_plots_4_5"], marker="o", label="NDMI plots 4,5")
ax.set_ylabel("NDMI")
ax.legend(loc="upper left")

# Cloud fraction (right axis; sparse dates)
ax2 = ax.twinx()
ax2.plot(df["date"], df["cloud_frac_terreno_casa"], linestyle="--", marker="x",
         alpha=0.6, label="Cloud frac (Terreno)")
ax2.set_ylabel("Cloud fraction")

# Precip (second right axis; ALL daily bars)
ax3 = ax.twinx()
ax3.spines["right"].set_position(("axes", 1.08))   # offset so both right axes are visible
ax3.set_frame_on(True); ax3.patch.set_visible(False)
ax3.bar(p["date"], p["precip_mm"].fillna(0),
        width=pd.Timedelta("0.85D"), alpha=0.3, label="Precip (mm)")
ax3.set_ylabel("Precip (mm)")

# Align x-limits to cover full precip range too
xmin = min(df["date"].min(), p["date"].min())
xmax = max(df["date"].max(), p["date"].max())
ax.set_xlim(xmin, xmax)

fig.autofmt_xdate()
fig.tight_layout()
plt.show()

KeyError: 'date'